In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv("dormmatch_cleaned.csv")

In [ ]:
numeric_features = ["sleep_duration","cleanliness_level","noise_tolerance","study_hours",
                   "social_activity","introvert_extrovert","guest_frequency","music_frequency"]
categorical_features = ["sharing_preference","weekend_activity","food_habit","smoking_preference",
                       "ac_preference","preferred_room_type","preferred_environment","roommate_interaction"]

In [ ]:
encoded = pd.get_dummies(df[numeric_features + categorical_features], columns=categorical_features)
scaler = StandardScaler()
X = scaler.fit_transform(encoded)
similarity_matrix = cosine_similarity(X)
similarity_matrix.shape

In [ ]:
def get_recommendations(student_index, top_n=5):
    scores = similarity_matrix[student_index].copy()
    scores[student_index] = -1
    top_indices = np.argsort(scores)[::-1][:top_n]
    results = df.iloc[top_indices].copy()
    results["compatibility_score"] = scores[top_indices]
    return results.sort_values("compatibility_score", ascending=False)

In [ ]:
get_recommendations(0, top_n=5)

In [ ]:
def explain_pair(student_idx, candidate_idx):
    a = df.iloc[student_idx]
    b = df.iloc[candidate_idx]
    comparisons = {}
    for feature in numeric_features:
        difference = abs(a[feature] - b[feature])
        comparisons[feature] = {"student":a[feature], "candidate":b[feature], "absolute_difference":difference}
    for feature in categorical_features:
        comparisons[feature] = {"student":a[feature], "candidate":b[feature], "match":a[feature] == b[feature]}
    return pd.DataFrame(comparisons).T

In [ ]:
candidate_index = get_recommendations(0, top_n=1).index[0]
explain_pair(0, candidate_index)

In [ ]:
recommendation_results = []
for idx in range(min(100, len(df))):
    recs = get_recommendations(idx, top_n=5)
    recommendation_results.append({"student_index":idx, "recommended_indices":recs.index.tolist()})
recommendation_results[:5]

In [ ]:
pd.DataFrame(recommendation_results).to_json("dormmatch_recommendations_sample.json", orient="records", indent=2)